# Size Estimation and Weight Prediction - Jetson Optimized
This notebook estimates pineapple size and predicts weight using ML models.

In [ ]:
import sys
sys.path.insert(0, '../config')
sys.path.insert(0, '..')

from paths import *
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle
import joblib

print('[OK] Libraries imported')

In [ ]:
# Load weight prediction data
print(f'Loading data from {WEIGHT_DATA_CSV}')
df = pd.read_csv(str(WEIGHT_DATA_CSV))

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nFirst few rows:')
print(df.head())
print(f'\nData info:')
print(df.info())
print(f'\nStatistics:')
print(df.describe())

In [ ]:
# Prepare data for training
# Assuming the CSV has pixel_count and weight columns
# Adjust column names based on your actual CSV structure

X = df.iloc[:, :-1].values  # Features (pixel counts)
y = df.iloc[:, -1].values   # Target (weight)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')
print(f'Features: {X_train.shape[1]}')

In [ ]:
# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Polynomial Regression': None,  # Will handle separately
    'SVR': SVR(kernel='rbf', C=100, epsilon=0.1),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=5)
}

results = {}

# Train Linear Regression
lr_model = models['Linear Regression']
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
results['Linear Regression'] = {
    'model': lr_model,
    'mse': mean_squared_error(y_test, y_pred_lr),
    'mae': mean_absolute_error(y_test, y_pred_lr),
    'r2': r2_score(y_test, y_pred_lr)
}

# Polynomial Regression
poly_features = PolynomialFeatures(degree=2)
X_train_poly = poly_features.fit_transform(X_train_scaled)
X_test_poly = poly_features.transform(X_test_scaled)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)
results['Polynomial Regression'] = {
    'model': poly_model,
    'poly_features': poly_features,
    'mse': mean_squared_error(y_test, y_pred_poly),
    'mae': mean_absolute_error(y_test, y_pred_poly),
    'r2': r2_score(y_test, y_pred_poly)
}

# Train other models (SVR, Decision Tree, KNN)
for model_name in ['SVR', 'Decision Tree', 'KNN']:
    model = models[model_name]
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[model_name] = {
        'model': model,
        'mse': mean_squared_error(y_test, y_pred),
        'mae': mean_absolute_error(y_test, y_pred),
        'r2': r2_score(y_test, y_pred)
    }

# Print results
print('\n=== Model Performance ===')
for model_name, metrics in results.items():
    print(f'\n{model_name}:')
    print(f'  MSE: {metrics["mse"]:.4f}')
    print(f'  MAE: {metrics["mae"]:.4f}')
    print(f'  R²: {metrics["r2"]:.4f}')

In [ ]:
# Save best model
best_model_name = max(results, key=lambda x: results[x]['r2'])
best_model_data = results[best_model_name]

print(f'\n✓ Best model: {best_model_name} (R² = {best_model_data["r2"]:.4f})')

# Save the best model and scaler
model_save_path = WEIGHT_MODEL
scaler_save_path = MODELS_DIR / 'weight_scaler.pkl'

joblib.dump(best_model_data['model'], str(model_save_path))
joblib.dump(scaler, str(scaler_save_path))

if 'poly_features' in best_model_data:
    joblib.dump(best_model_data['poly_features'], str(MODELS_DIR / 'weight_poly_features.pkl'))
    print(f'✓ Saved: Polynomial features')

print(f'✓ Model saved to {model_save_path}')
print(f'✓ Scaler saved to {scaler_save_path}')
print(f'\nModels directory:')
for f in MODELS_DIR.glob('weight*'):
    print(f'  - {f.name} ({f.stat().st_size / 1024:.2f} KB)')

In [ ]:
# Test prediction function
def predict_weight(pixel_count):
    """Predict pineapple weight from pixel count"""
    loaded_model = joblib.load(str(WEIGHT_MODEL))
    loaded_scaler = joblib.load(str(SCALER_SAVE_PATH))
    
    pixel_count_scaled = loaded_scaler.transform([[pixel_count]])
    weight = loaded_model.predict(pixel_count_scaled)[0]
    return weight

# Test with sample
print('Testing prediction functions...')
sample_pixel = X_test[0][0]
predicted_weight = predict_weight(sample_pixel)
actual_weight = y_test[0]

print(f'Sample prediction:')
print(f'  Pixel count: {sample_pixel:.0f}')
print(f'  Predicted weight: {predicted_weight:.2f}kg')
print(f'  Actual weight: {actual_weight:.2f}kg')
print(f'  Error: {abs(predicted_weight - actual_weight):.2f}kg')

## Summary
✓ Weight prediction models trained
✓ Best model saved and ready for deployment
✓ Can integrate with detection/classification pipeline